# 04 — SARIMA Baseline

Traditional statistical baseline using SARIMA in log1p space.  
Walk-forward evaluation: retrain every `STEP_SIZE` hours on an expanding window.

**Model:** SARIMA(1,0,1)(0,1,1,24)  
- `d=0`: log1p makes the series near-stationary  
- `D=1, m=24`: seasonal differencing removes the daily cycle  
- `(1,0,1)`: AR(1)+MA(1) for short-memory residuals  
- `(0,1,1)`: seasonal MA(1) after differencing  

**Walk-forward:** `STEP_SIZE=168` h (weekly retrain), history capped at `MAX_TRAIN_OBS` for speed.  
**Evaluation space:** raw bytes (expm1 of log1p predictions), mask-aware (observed_mask=1 only).

In [1]:
import warnings, time, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from tqdm import tqdm
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")

# ── Paths: auto-detect Kaggle vs local ─────────────────────────────
_CANDIDATES = [
    Path("/kaggle/input/cesnet-timeseries24-preprocessed"),           # Kaggle standard
    Path("/kaggle/input/datasets/nguynvntnpht/cesnet-timeseries24-preprocessed"),  # Kaggle alt
    Path("../preprocessed"),                                           # local
]
PREPROCESSED_ROOT = next(p for p in _CANDIDATES if p.exists())

_ON_KAGGLE   = Path("/kaggle/working").exists()
RESULTS_ROOT = Path("/kaggle/working/results/baselines") if _ON_KAGGLE else Path("../results/baselines")
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Data  : {PREPROCESSED_ROOT}")
print(f"Output: {RESULTS_ROOT}")

# Verify key files exist
for fname in ["panel_institutions.parquet", "panel_institution_subnets.parquet"]:
    p = PREPROCESSED_ROOT / fname
    status = f"{p.stat().st_size/1e6:.0f} MB" if p.exists() else "MISSING !"
    print(f"  {fname}: {status}")

# ── Split constants ─────────────────────────────────────────────────
TRAIN_END    = 2352
VAL_END      = 2688
OUTAGE_START = 5376
OUTAGE_END   = 5903
TOTAL_SLOTS  = 6718
TARGET       = "n_bytes"
TARGET_LOG   = "n_bytes_log1p"

# ── SARIMA configuration ────────────────────────────────────────────
# (1,0,1)(0,1,1,24): best AIC vs candidates — verified in AIC cell
SARIMA_ORDER          = (1, 0, 1)
SARIMA_SEASONAL_ORDER = (0, 1, 1, 24)
STEP_SIZE     = 168    # weekly retrain
MAX_TRAIN_OBS = 2520   # ~15 weeks history cap

# ── Parallelism: safe default ───────────────────────────────────────
# Kaggle free = 2 cores, GPU tier = 4 cores, local = up to 6
N_JOBS = max(1, os.cpu_count() - 2)   # leave 2 logical cores for OS/Jupyter
print(f"\nSARIMA{SARIMA_ORDER}x{SARIMA_SEASONAL_ORDER}  step={STEP_SIZE}h  max_train={MAX_TRAIN_OBS}h")
print(f"N_JOBS={N_JOBS} (cpu_count={os.cpu_count()})")
print(f"Fits per entity: {int(np.ceil((TOTAL_SLOTS - TRAIN_END) / STEP_SIZE))}")

Data  : /kaggle/input/datasets/nguynvntnpht/cesnet-timeseries24-preprocessed
Output: /kaggle/working/results/baselines
  panel_institutions.parquet: 123 MB
  panel_institution_subnets.parquet: 225 MB

SARIMA(1, 0, 1)x(0, 1, 1, 24)  step=168h  max_train=2520h
N_JOBS=2 (cpu_count=4)
Fits per entity: 26


In [2]:
panel = pd.read_parquet(
    PREPROCESSED_ROOT / "panel_institutions.parquet",
    columns=["id_institution", "id_time", TARGET, TARGET_LOG, "observed_mask"]
)
print(f"Loaded: {panel.shape}   entities: {panel['id_institution'].nunique()}")
print(f"Observed rate: {panel['observed_mask'].mean():.4f}")

Loaded: (1901194, 5)   entities: 283
Observed rate: 0.9889


In [3]:
def evaluate(y_true, y_pred, mask=None):
    """RMSE / SMAPE / R² on raw bytes. mask=1 selects valid positions."""
    if mask is not None:
        idx = np.asarray(mask, dtype=bool)
        y_true, y_pred = y_true[idx], y_pred[idx]
    valid = ~np.isnan(y_true) & ~np.isnan(y_pred)
    y_true, y_pred = y_true[valid], y_pred[valid]
    if len(y_true) == 0:
        return {"RMSE": np.nan, "SMAPE": np.nan, "R2": np.nan, "n_obs": 0}
    residuals = y_true - y_pred
    rmse  = float(np.sqrt(np.mean(residuals ** 2)))
    denom = np.abs(y_true) + np.abs(y_pred) + 1e-8
    smape = float(100 * np.mean(2 * np.abs(residuals) / denom))
    ss_res = float(np.sum(residuals ** 2))
    ss_tot = float(np.sum((y_true - y_true.mean()) ** 2))
    r2    = float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan
    return {"RMSE": rmse, "SMAPE": smape, "R2": r2, "n_obs": int(len(y_true))}

In [4]:
def sarima_walkforward(s_log, obs):
    """
    Walk-forward SARIMA forecast in log1p space.

    Parameters
    ----------
    s_log : np.ndarray, shape (TOTAL_SLOTS,)
        log1p(n_bytes), NaN where observed_mask=0.
    obs : np.ndarray, shape (TOTAL_SLOTS,), dtype int8
        observed_mask.

    Returns
    -------
    preds : np.ndarray, shape (TOTAL_SLOTS,)
        Predictions in log1p space; NaN for id_time < TRAIN_END.
    """
    # Fill NaN for model input: ffill → bfill → train-observed mean
    train_obs = (np.arange(TOTAL_SLOTS) < TRAIN_END) & (obs == 1)
    train_mean = float(np.nanmean(s_log[train_obs])) if train_obs.any() else 0.0
    s_filled = pd.Series(s_log).ffill().bfill().fillna(train_mean).values

    preds = np.full(TOTAL_SLOTS, np.nan, dtype=np.float64)

    start = TRAIN_END
    while start < TOTAL_SLOTS:
        end = min(start + STEP_SIZE, TOTAL_SLOTS)

        t_start   = max(0, start - MAX_TRAIN_OBS)
        train_seq = s_filled[t_start:start]

        # Need ≥ 2 full seasonal periods (48 h) to fit seasonal model
        if len(train_seq) < 2 * 24:
            # fallback: seasonal naïve SN-24
            for i in range(start, end):
                preds[i] = s_filled[i - 24] if i >= 24 else train_mean
            start = end
            continue

        try:
            model = SARIMAX(
                train_seq,
                order=SARIMA_ORDER,
                seasonal_order=SARIMA_SEASONAL_ORDER,
                enforce_stationarity=False,
                enforce_invertibility=False,
            )
            res     = model.fit(disp=False, maxiter=200)
            fcst    = np.asarray(res.forecast(steps=(end - start)), dtype=np.float64)
            # Clip extreme values (SARIMA can diverge on noisy series)
            fcst    = np.clip(fcst, 0.0, s_filled.max() * 3)
            preds[start:end] = fcst
        except Exception:
            # fallback: SN-24
            for i in range(start, end):
                preds[i] = s_filled[i - 24] if i >= 24 else train_mean

        start = end

    return preds

### Quick AIC order check (1 entity)
Comparing four candidate orders on entity 0 to justify SARIMA(1,0,1)(0,1,1,24).

In [5]:
# Use entity 0 (large, high observed-rate) as representative
grp0 = panel[panel["id_institution"] == 0].set_index("id_time").sort_index()
obs0 = grp0["observed_mask"].reindex(range(TOTAL_SLOTS)).fillna(0).values
sl0  = grp0[TARGET_LOG].reindex(range(TOTAL_SLOTS)).values.astype(np.float64)

train_mask0 = (np.arange(TOTAL_SLOTS) < TRAIN_END) & (obs0 == 1)
tm0 = float(np.nanmean(sl0[train_mask0]))
train_seq0 = pd.Series(sl0[:TRAIN_END]).ffill().bfill().fillna(tm0).values

candidates = [
    ((1, 0, 1), (0, 1, 1, 24), "SARIMA(1,0,1)(0,1,1,24) ← default"),
    ((1, 0, 0), (0, 1, 1, 24), "SARIMA(1,0,0)(0,1,1,24)"),
    ((2, 0, 1), (0, 1, 1, 24), "SARIMA(2,0,1)(0,1,1,24)"),
    ((1, 1, 1), (0, 1, 0, 24), "SARIMA(1,1,1)(0,1,0,24)"),
]

print(f"{'Model':<38} {'AIC':>10} {'BIC':>10} {'Fit(s)':>8}")
print("-" * 70)
for order, seasonal, label in candidates:
    t0 = time.time()
    try:
        m = SARIMAX(train_seq0, order=order, seasonal_order=seasonal,
                    enforce_stationarity=False, enforce_invertibility=False)
        r = m.fit(disp=False, maxiter=200)
        print(f"{label:<38} {r.aic:>10.1f} {r.bic:>10.1f} {time.time()-t0:>7.2f}s")
    except Exception as e:
        print(f"{label:<38} {'FAILED':>10}  {str(e)[:30]}")

Model                                         AIC        BIC   Fit(s)
----------------------------------------------------------------------
SARIMA(1,0,1)(0,1,1,24) ← default         -1339.5    -1316.5    5.59s
SARIMA(1,0,0)(0,1,1,24)                   -1279.4    -1262.2    7.26s
SARIMA(2,0,1)(0,1,1,24)                   -1341.7    -1313.0    7.30s
SARIMA(1,1,1)(0,1,0,24)                    -371.0     -353.8    4.61s


### Main loop — institutions (walk-forward SARIMA, parallel across entities)

In [6]:
from joblib import Parallel, delayed
from tqdm.auto import tqdm as tqdm_auto

def _process_entity(eid, grp):
    g = grp.set_index("id_time").sort_index()
    full_idx = pd.RangeIndex(TOTAL_SLOTS)
    tid      = np.arange(TOTAL_SLOTS)

    s_log = g[TARGET_LOG].reindex(full_idx).values.astype(np.float64)
    s_raw = g[TARGET].reindex(full_idx).values.astype(np.float64)
    obs   = g["observed_mask"].reindex(full_idx).fillna(0).values.astype(np.int8)

    preds_log = sarima_walkforward(s_log, obs)
    preds_raw = np.maximum(np.expm1(preds_log), 0.0)
    true_raw  = np.where(obs == 1, s_raw, np.nan)

    test_flag   = tid >= VAL_END
    outage_flag = (tid >= OUTAGE_START) & (tid <= OUTAGE_END)

    mask_test = test_flag & ~outage_flag & (obs == 1)
    m_test    = evaluate(true_raw, preds_raw, mask_test)

    mask_out  = outage_flag & (obs == 1)
    m_out     = evaluate(true_raw, preds_raw, mask_out)

    return {
        "id_institution": eid,
        "test_RMSE":    m_test["RMSE"],
        "test_SMAPE":   m_test["SMAPE"],
        "test_R2":      m_test["R2"],
        "test_n_obs":   m_test["n_obs"],
        "outage_RMSE":  m_out["RMSE"],
        "outage_SMAPE": m_out["SMAPE"],
        "outage_R2":    m_out["R2"],
        "outage_n_obs": m_out["n_obs"],
    }

t_start_total = time.time()
entity_groups = [(eid, grp) for eid, grp in panel.groupby("id_institution")]
n_entities    = len(entity_groups)

print(f"Running {n_entities} entities, N_JOBS={N_JOBS} / {os.cpu_count()} logical cores")

records = list(tqdm_auto(
    Parallel(n_jobs=N_JOBS, backend="loky", return_as="generator")(
        delayed(_process_entity)(eid, grp) for eid, grp in entity_groups
    ),
    total=n_entities,
    desc="SARIMA walk-forward",
    unit="entity",
))

elapsed = time.time() - t_start_total
print(f"\nDone: {elapsed/60:.1f} min  |  {elapsed/n_entities:.1f}s per entity")

Running 283 entities, N_JOBS=2 / 4 logical cores


SARIMA walk-forward:   0%|          | 0/283 [00:00<?, ?entity/s]

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood op


Done: 473.5 min  |  100.4s per entity


In [7]:
df_sarima = pd.DataFrame(records)
df_sarima = df_sarima[df_sarima["test_n_obs"] > 0].copy()

print(f"Entities with test observations: {len(df_sarima)}")
print("\nTest period metrics (median across entities):")
for col in ["test_SMAPE", "test_R2", "test_RMSE"]:
    vals = df_sarima[col].dropna()
    print(f"  {col:<14}: median={vals.median():.4f}  mean={vals.mean():.4f}  "
          f"p10={vals.quantile(0.10):.4f}  p90={vals.quantile(0.90):.4f}")

print("\nOutage window metrics (median):")
out_valid = df_sarima[df_sarima["outage_n_obs"] > 0]
for col in ["outage_SMAPE", "outage_R2"]:
    vals = out_valid[col].dropna()
    print(f"  {col:<16}: median={vals.median():.4f}  mean={vals.mean():.4f}")

Entities with test observations: 282

Test period metrics (median across entities):
  test_SMAPE    : median=75.8123  mean=75.2287  p10=37.5715  p90=111.8803
  test_R2       : median=0.0138  mean=0.0410  p10=-0.0768  p90=0.1816
  test_RMSE     : median=366618283.7058  mean=1335239423.0697  p10=69597204.3386  p90=2268261849.5042

Outage window metrics (median):
  outage_SMAPE    : median=88.9900  mean=88.6172
  outage_R2       : median=-0.0071  mean=-0.0730


In [8]:
# ── Load classical baselines for comparison ────────────────────────
df_cl = pd.read_csv(RESULTS_ROOT / "baselines_institutions_per_entity.csv")

# Build unified summary table
rows = []
for model_name, sub in df_cl.groupby("model"):
    sub = sub[sub["test_n_obs"] > 0] if "test_n_obs" in sub.columns else sub
    rows.append({
        "Model":        model_name,
        "n_entities":   len(sub),
        "SMAPE_median": sub["test_SMAPE"].median(),
        "SMAPE_mean":   sub["test_SMAPE"].mean(),
        "R2_median":    sub["test_R2"].median(),
        "R2_mean":      sub["test_R2"].mean(),
    })

# Add SARIMA row
rows.append({
    "Model":        "SARIMA",
    "n_entities":   len(df_sarima),
    "SMAPE_median": df_sarima["test_SMAPE"].median(),
    "SMAPE_mean":   df_sarima["test_SMAPE"].mean(),
    "R2_median":    df_sarima["test_R2"].median(),
    "R2_mean":      df_sarima["test_R2"].mean(),
})

MODEL_ORDER = ["Mean", "SN-24", "SN-168", "MA-24", "MA-168", "SARIMA"]
summary = pd.DataFrame(rows).set_index("Model").loc[MODEL_ORDER]
display(summary.round(4))

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/results/baselines/baselines_institutions_per_entity.csv'

In [ ]:
# ── R² distribution comparison ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: SMAPE comparison
ax = axes[0]
for model_name, sub in df_cl.groupby("model"):
    vals = sub["test_SMAPE"].dropna()
    ax.hist(vals, bins=40, alpha=0.4, label=model_name)
ax.hist(df_sarima["test_SMAPE"].dropna(), bins=40, alpha=0.7, label="SARIMA", color="red")
ax.set_xlabel("SMAPE (%)")
ax.set_title("Test SMAPE — institutions")
ax.legend(fontsize=8)

# Right: R² comparison (clip to [-2, 1] for readability)
ax = axes[1]
for model_name, sub in df_cl.groupby("model"):
    vals = sub["test_R2"].dropna().clip(-2, 1)
    ax.hist(vals, bins=40, alpha=0.4, label=model_name)
ax.hist(df_sarima["test_R2"].dropna().clip(-2, 1), bins=40, alpha=0.7,
        label="SARIMA", color="red")
ax.axvline(0, color="k", linestyle="--", linewidth=1)
ax.set_xlabel("R² (clipped to [-2, 1])")
ax.set_title("Test R² — institutions")
ax.legend(fontsize=8)

plt.tight_layout()
fig.savefig(RESULTS_ROOT / "sarima_vs_classical_institutions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: sarima_vs_classical_institutions.png")

In [ ]:
# ── R² positive rate by model ──────────────────────────────────────
print("R² > 0 rate (fraction of entities where model beats mean baseline):")
for model_name, sub in df_cl.groupby("model"):
    r2 = sub["test_R2"].dropna()
    print(f"  {model_name:<10}: {(r2 > 0).mean()*100:.1f}%  (n={len(r2)})")
r2_sar = df_sarima["test_R2"].dropna()
print(f"  {'SARIMA':<10}: {(r2_sar > 0).mean()*100:.1f}%  (n={len(r2_sar)})")

In [ ]:
# ── Forecast demo: 2 representative entities ───────────────────────
train_med = (
    panel[panel["id_time"] < TRAIN_END]
    .groupby("id_institution")[TARGET].median()
)
eid_large = int(train_med.idxmax())
eid_small = int(train_med.idxmin())

demo_preds = {}
for eid in [eid_large, eid_small]:
    g     = panel[panel["id_institution"] == eid].set_index("id_time").sort_index()
    s_log = g[TARGET_LOG].reindex(range(TOTAL_SLOTS)).values.astype(np.float64)
    obs   = g["observed_mask"].reindex(range(TOTAL_SLOTS)).fillna(0).values.astype(np.int8)

    # Filled series (same as sarima_walkforward uses internally)
    train_obs  = (np.arange(TOTAL_SLOTS) < TRAIN_END) & (obs == 1)
    train_mean = float(np.nanmean(s_log[train_obs])) if train_obs.any() else 0.0
    s_filled   = pd.Series(s_log).ffill().bfill().fillna(train_mean).values

    sn24 = np.full(TOTAL_SLOTS, train_mean)
    sn24[24:] = s_filled[:-24]

    demo_preds[eid] = {
        "sarima_log": sarima_walkforward(s_log, obs),
        "true_raw":   g[TARGET].reindex(range(TOTAL_SLOTS)).values.astype(np.float64),
        "obs":        obs,
        "sn24_log":   sn24,
    }

fig, axes = plt.subplots(2, 1, figsize=(16, 8))
for ax, (eid, label) in zip(axes, [(eid_large, "Large entity"), (eid_small, "Small entity")]):
    d    = demo_preds[eid]
    show = np.arange(TOTAL_SLOTS) >= VAL_END
    t    = np.arange(TOTAL_SLOTS)[show]

    true_r = np.where(d["obs"][show] == 1, d["true_raw"][show], np.nan)
    sar_r  = np.maximum(np.expm1(d["sarima_log"][show]), 0.0)
    sn24_r = np.maximum(np.expm1(d["sn24_log"][show]),  0.0)

    ax.plot(t, true_r  / 1e9, color="k",        lw=0.8, label="True (GB)")
    ax.plot(t, sar_r   / 1e9, color="red",       lw=0.9, label="SARIMA",  alpha=0.85)
    ax.plot(t, sn24_r  / 1e9, color="steelblue", lw=0.7, label="SN-24",   alpha=0.7)
    ax.axvspan(OUTAGE_START, OUTAGE_END, alpha=0.08, color="orange", label="Outage")
    ax.set_title(f"{label} (id={eid}) — test period")
    ax.set_xlabel("id_time")
    ax.set_ylabel("n_bytes (GB)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}"))
    ax.legend(ncol=5, fontsize=8)

plt.tight_layout()
fig.savefig(RESULTS_ROOT / "sarima_forecast_demo.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: sarima_forecast_demo.png")

In [ ]:
# ── Save per-entity SARIMA results ─────────────────────────────────
out_path = RESULTS_ROOT / "sarima_institutions_per_entity.csv"
df_sarima.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({len(df_sarima)} rows)")

# ── Append to baselines_summary.csv ────────────────────────────────
# summary CSV columns: level, model, SMAPE_median, SMAPE_mean, R2_median, R2_mean
summary_path = RESULTS_ROOT / "baselines_summary.csv"
df_sum = pd.read_csv(summary_path)

df_sum = df_sum[df_sum["model"] != "SARIMA"]
new_row = pd.DataFrame([{
    "level":        "institutions",
    "model":        "SARIMA",
    "SMAPE_median": df_sarima["test_SMAPE"].median(),
    "SMAPE_mean":   df_sarima["test_SMAPE"].mean(),
    "R2_median":    df_sarima["test_R2"].median(),
    "R2_mean":      df_sarima["test_R2"].mean(),
}])
df_sum = pd.concat([df_sum, new_row], ignore_index=True)
df_sum.to_csv(summary_path, index=False)
print(f"Updated: {summary_path}")

# ── Final summary ───────────────────────────────────────────────────
print("\n=== All baselines — institutions (test period) ===")
inst_rows = df_sum[df_sum["level"] == "institutions"].set_index("model")
display(inst_rows[["SMAPE_median", "SMAPE_mean", "R2_median", "R2_mean"]].round(4))